# Stage 5b. End-to-End Fine-tuning of CSA

Notebook ini memperbaiki temuan bahwa modul CSA dan layer proyeksi pada backbone deep embedding Stage 4/5 tidak pernah dilatih (murni inisialisasi acak) karena embedding dibekukan sebagai fitur tetap. Di sini, badan ResNet18 dibekukan (bobot ImageNet dipertahankan), sedangkan CSA, proyeksi, fusion attention, trunk, dan tiga head dilatih bersama lewat backpropagation penuh atas citra. Hyperparameter memakai hasil pencarian Optuna Stage 5 sebagai titik awal, bukan pencarian ulang dari nol.

## Environment Setup

In [1]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent.parent))

import json

import numpy as np
import pandas as pd
import torch

from configs import paths
from src.common import features, manifest as manifest_utils, train
from src.sites.conjunctiva import data

output_dir = paths.outputs_dir("conjunctiva")
artifact_dir = paths.artifacts_dir("conjunctiva")

manifest = manifest_utils.assign_kfold(data.build_manifest(save=False), n_splits=5, seed=42)
handcrafted = pd.read_csv(output_dir / "handcrafted_features.csv")
with open(output_dir / "multitask_optuna_best_params.json") as handle:
    best_params = json.load(handle)["best_params"]

print("manifest", manifest.shape)
print("best params (titik awal)", best_params)
print("device", "cuda" if torch.cuda.is_available() else "cpu")

Eyes-Defy Italy melewati 2 folder tanpa metadata atau file lengkap: [93, 95]
Eyes-Defy India melewati 1 folder tanpa metadata atau file lengkap: [7]
manifest (925, 14)
best params (titik awal) {'learning_rate': 0.003055737416493421, 'epochs': 84, 'weight_classification': 1.9482952299704612, 'weight_severity': 0.3896833153575715, 'focal_gamma': 2.0367179391695727, 'dropout': 0.3119141432082831, 'trunk_dim': 256, 'attention_dim': 32, 'batch_size': 64}
device cuda


## Train End-to-End (Backbone Dibekukan)

Ukuran batch dikurangi dibanding pelatihan tabular karena sekarang memuat citra, bukan vektor fitur kecil. Epoch dan bobot loss lainnya memakai hasil tuning Stage 5 sebagai titik awal.

In [2]:
result = train.run_kfold_end_to_end(
    manifest, handcrafted,
    n_splits=5,
    epochs=best_params["epochs"],
    batch_size=32,
    learning_rate=best_params["learning_rate"],
    loss_weights=(1.0, best_params["weight_classification"], best_params["weight_severity"]),
    trunk_dim=best_params["trunk_dim"],
    attention_dim=best_params["attention_dim"],
    dropout=best_params["dropout"],
    focal_gamma=best_params["focal_gamma"],
    backbone_name="resnet18",
)
print(result["fold_metrics"].round(4).to_string(index=False))

 fold  n_val    mae   rmse  accuracy  severity_accuracy
    0    186 1.8697 2.3822    0.6290             0.3357
    1    185 1.9295 2.3985    0.4432             0.3310
    2    185 1.6700 2.1330    0.5892             0.3028
    3    185 1.7037 2.2059    0.6595             0.3310
    4    184 1.7957 2.2941    0.6304             0.3262


## Verify CSA Actually Learned

Memastikan badan backbone tetap beku dan CSA benar-benar menerima gradien, bukan sekadar klaim.

In [3]:
sample_model = result["models"][0]
backbone = sample_model.embedding_backbone
print("early requires_grad (harus False)", next(backbone.early.parameters()).requires_grad)
print("late requires_grad (harus False)", next(backbone.late.parameters()).requires_grad)
print("csa requires_grad (harus True)", next(backbone.csa.parameters()).requires_grad)
print("projection requires_grad (harus True)", backbone.projection.weight.requires_grad)

fresh_backbone = features.EmbeddingBackbone()
csa_weight_diff = (
    backbone.csa.channel_attention.mlp[0].weight.cpu() - fresh_backbone.csa.channel_attention.mlp[0].weight
).abs().mean().item()
print("rerata selisih bobot CSA terlatih vs inisialisasi acak baru (harus jauh dari nol)", round(csa_weight_diff, 6))

early requires_grad (harus False) False
late requires_grad (harus False) False
csa requires_grad (harus True) True
projection requires_grad (harus True) True
rerata selisih bobot CSA terlatih vs inisialisasi acak baru (harus jauh dari nol) 0.088955


## Compare with Tabular Baseline

Dibandingkan jujur dengan hasil Stage 5 (CSA belum terlatih, embedding dibekukan), apa pun hasilnya.

In [4]:
baseline_fold_metrics = pd.read_csv(output_dir / "multitask_fold_metrics_full_fusion_resnet18_csa_tuned.csv")
comparison = pd.DataFrame([
    {
        "configuration": "Tabular (embedding dibekukan, CSA belum terlatih)",
        "mae_mean": baseline_fold_metrics["mae"].mean(),
        "accuracy_mean": baseline_fold_metrics["accuracy"].mean(),
        "severity_accuracy_mean": baseline_fold_metrics["severity_accuracy"].mean(),
    },
    {
        "configuration": "End-to-end (CSA terlatih, backbone dibekukan)",
        "mae_mean": result["fold_metrics"]["mae"].mean(),
        "accuracy_mean": result["fold_metrics"]["accuracy"].mean(),
        "severity_accuracy_mean": result["fold_metrics"]["severity_accuracy"].mean(),
    },
])
print(comparison.round(4).to_string(index=False))

breakdown = train.evaluate_by_dataset(result["oof"], manifest)
print()
print("breakdown per dataset (end-to-end):")
print(breakdown.round(4).to_string(index=False))

                                    configuration  mae_mean  accuracy_mean  severity_accuracy_mean
Tabular (embedding dibekukan, CSA belum terlatih)    1.5538         0.7329                  0.4155
    End-to-end (CSA terlatih, backbone dibekukan)    1.7937         0.5903                  0.3253

breakdown per dataset (end-to-end):
  dataset   n    mae  accuracy  severity_accuracy
cp_anemic 710 1.7988    0.5732             0.3254
eyes_defy 215 1.7772    0.6465                NaN


## Save Results

In [5]:
comparison.to_csv(output_dir / "multitask_end_to_end_comparison.csv", index=False)
result["oof"].to_csv(output_dir / "multitask_oof_end_to_end.csv", index=False)
result["fold_metrics"].to_csv(output_dir / "multitask_fold_metrics_end_to_end.csv", index=False)

for fold_index, fold_model in enumerate(result["models"]):
    checkpoint_path = artifact_dir / f"end_to_end_fold{fold_index}.pt"
    torch.save(fold_model.state_dict(), checkpoint_path)

print("saved end-to-end results and checkpoints to", output_dir, "and", artifact_dir)

saved end-to-end results and checkpoints to /home/praktikan/projects/Azril/hemavision/outputs/conjunctiva and /home/praktikan/projects/Azril/hemavision/artifacts/conjunctiva
